# Lab 6: Multi-Agent Coding Team for ERP Development
## CPE494: ERP Systems with AI Integration

This notebook is the local VS Code orchestrator for the **CPE494-agent-coding-team** repository.

It coordinates four Gemini-powered agents:

1. **Architect**: creates the sprint blueprint and task list.
2. **Coder**: generates source code and project artifacts.
3. **Logic Tester**: audits code for data, transaction, security, and compilation risks.
4. **UI Auditor**: audits UI/UX against the Zen Green reference rules.

The generated application lives in the companion repository:

```text
../CPE494-erp-invoice-app-by-ai
```

This notebook intentionally uses plain Python first. Later, students can convert the workflow to use ADK and MCP. Apparently we must first understand the machine before adding more machinery to it.


## Step 0: Expected folder structure

The two repositories should be cloned side-by-side inside the parent folder `lab-coding-agent`:

```text
lab-coding-agent/
  CPE494-agent-coding-team/
    notebooks/
      01_agent_demo.ipynb
    prompts/
      role_architect.txt
      role_coder.txt
      role_logic_tester.txt
      role_ui_auditor.txt
    logic_specs/
      erp_invoice_logic.md
    ui_specs/
      erp_invoice_ui.md
      ui_reference.pdf
    outputs/
    .env

  CPE494-erp-invoice-app-by-ai/
    Pages/
    Models/
    Data/
    wwwroot/
```

The `.env` file belongs in the root of `CPE494-agent-coding-team` and should contain:

```text
GOOGLE_API_KEY=your_api_key_here
```

Make sure `.env` is listed in `.gitignore`. Leaking API keys to GitHub is an educational experience, but not the kind we need.


In [ ]:
# Step 1: Imports and path setup
from pathlib import Path
import json
import os
import re
import shutil
import subprocess
from datetime import datetime

from dotenv import load_dotenv
from google import genai

# The notebook is expected to run from the CPE494-agent-coding-team repo root.
# In VS Code, set the notebook working directory to the repository root if needed.
REPO_ROOT = Path.cwd().resolve()

# If the notebook is opened from notebooks/, move one level up automatically.
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent.resolve()

TARGET_APP_PATH = (REPO_ROOT.parent / "CPE494-erp-invoice-app-by-ai").resolve()

PROMPTS_DIR = REPO_ROOT / "prompts"
LOGIC_SPECS_DIR = REPO_ROOT / "logic_specs"
UI_SPECS_DIR = REPO_ROOT / "ui_specs"
OUTPUTS_DIR = REPO_ROOT / "outputs"
GENERATED_DIR = OUTPUTS_DIR / "generated_files"
AUDIT_LOG_PATH = OUTPUTS_DIR / "audit_log.jsonl"
MANIFEST_PATH = OUTPUTS_DIR / "latest_manifest.md"
TASKS_PATH = OUTPUTS_DIR / "latest_tasks.json"

OUTPUTS_DIR.mkdir(exist_ok=True)
GENERATED_DIR.mkdir(exist_ok=True)

print("Agent repo root:", REPO_ROOT)
print("Target app path:", TARGET_APP_PATH)


In [ ]:
# Step 2: Validate required folders and files
required_paths = [
    PROMPTS_DIR / "role_architect.txt",
    PROMPTS_DIR / "role_coder.txt",
    PROMPTS_DIR / "role_logic_tester.txt",
    PROMPTS_DIR / "role_ui_auditor.txt",
    LOGIC_SPECS_DIR / "erp_invoice_logic.md",
    UI_SPECS_DIR / "erp_invoice_ui.md",
]

missing = [p for p in required_paths if not p.exists()]

if not TARGET_APP_PATH.exists():
    missing.append(TARGET_APP_PATH)

if missing:
    print("Missing required paths:")
    for p in missing:
        print(" -", p)
    raise FileNotFoundError("Fix the missing files/folders before continuing.")

print("All required folders and files were found.")


In [ ]:
# Step 3: Load environment variables and initialize Gemini client
load_dotenv(REPO_ROOT / ".env")

api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("GOOGLE_API_KEY was not found. Create a .env file in the agent repo root.")

MODEL_NAME = os.getenv("GEMINI_MODEL", "gemini-2.5-pro")
client = genai.Client(api_key=api_key)

print("Gemini client initialized.")
print("Model:", MODEL_NAME)


In [ ]:
# Step 4: Load agent prompts and specification documents

def read_text(path: Path) -> str:
    return path.read_text(encoding="utf-8")

ARCHITECT_PROMPT = read_text(PROMPTS_DIR / "role_architect.txt")
CODER_PROMPT = read_text(PROMPTS_DIR / "role_coder.txt")
LOGIC_TESTER_PROMPT = read_text(PROMPTS_DIR / "role_logic_tester.txt")
UI_AUDITOR_PROMPT = read_text(PROMPTS_DIR / "role_ui_auditor.txt")

LOGIC_SPECS = read_text(LOGIC_SPECS_DIR / "erp_invoice_logic.md")
UI_SPECS = read_text(UI_SPECS_DIR / "erp_invoice_ui.md")

print("Loaded prompts and specs.")
print("Logic spec length:", len(LOGIC_SPECS))
print("UI spec length:", len(UI_SPECS))


In [ ]:
# Step 5: Utility functions

def call_agent(system_prompt: str, user_prompt: str, temperature: float = 0.2) -> str:
    """Call Gemini with a system prompt and user prompt."""
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=user_prompt,
        config={
            "system_instruction": system_prompt,
            "temperature": temperature,
        },
    )
    return response.text or ""


def strip_markdown_code_fence(text: str) -> str:
    """Remove accidental markdown fences from coder output."""
    cleaned = text.strip()
    fence = re.match(r"^```[a-zA-Z0-9_+-]*\s*(.*?)\s*```$", cleaned, re.DOTALL)
    if fence:
        return fence.group(1).strip()
    return cleaned


def starts_with_pass(text: str) -> bool:
    """Auditors must start with PASS to approve."""
    return text.strip().upper().startswith("PASS")


def safe_relative_path(file_name: str) -> Path:
    """Return a safe relative path and prevent path traversal."""
    raw = file_name.replace("\", "/").strip().lstrip("/")
    rel = Path(raw)
    if rel.is_absolute() or ".." in rel.parts:
        raise ValueError(f"Unsafe file path from task: {file_name}")
    return rel


def get_project_tree(root: Path, max_files: int = 120) -> str:
    """Return a compact project tree for agent context."""
    ignore_dirs = {".git", "bin", "obj", ".vs", ".vscode", "node_modules"}
    files = []
    for path in root.rglob("*"):
        if any(part in ignore_dirs for part in path.parts):
            continue
        if path.is_file():
            try:
                rel = path.relative_to(root)
                files.append(str(rel).replace("\", "/"))
            except ValueError:
                pass
        if len(files) >= max_files:
            break
    return "
".join(sorted(files))


def read_existing_target_file(relative_path: Path) -> str:
    """Read existing target file if it exists."""
    target_file = TARGET_APP_PATH / relative_path
    if target_file.exists() and target_file.is_file():
        return target_file.read_text(encoding="utf-8", errors="replace")
    return ""


def save_generated_file(relative_path: Path, content: str) -> Path:
    """Save generated file under outputs/generated_files using its target relative path."""
    destination = GENERATED_DIR / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(content, encoding="utf-8")
    return destination


def append_audit_log(record: dict):
    record = {"timestamp": datetime.now().isoformat(timespec="seconds"), **record}
    with AUDIT_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "
")


In [ ]:
# Step 6: Architect agent

def extract_tasks_from_manifest(manifest_text: str) -> list[dict]:
    """Extract JSON task list from the Architect response."""
    json_match = re.search(r"```json\s*(.*?)\s*```", manifest_text, re.DOTALL | re.IGNORECASE)
    if json_match:
        raw_json = json_match.group(1)
    else:
        # Fallback: find the first JSON object in the response.
        obj_match = re.search(r"\{.*\}", manifest_text, re.DOTALL)
        raw_json = obj_match.group(0) if obj_match else ""

    if not raw_json:
        return []

    try:
        parsed = json.loads(raw_json)
    except json.JSONDecodeError as e:
        print("Could not parse Architect JSON:", e)
        return []

    tasks = parsed.get("tasks", [])
    valid_tasks = []
    for i, task in enumerate(tasks, start=1):
        file_name = task.get("file_name")
        task_description = task.get("task_description")
        if file_name and task_description:
            valid_tasks.append({
                "task_id": i,
                "file_name": file_name,
                "task_description": task_description,
            })
    return valid_tasks


def call_architect(sprint_goal: str) -> tuple[str, list[dict]]:
    """Ask the Architect to create a sprint plan and task list."""
    project_tree = get_project_tree(TARGET_APP_PATH)

    user_prompt = f"""
You are planning work for the target ASP.NET Core project below.

TARGET PROJECT PATH:
{TARGET_APP_PATH}

CURRENT TARGET PROJECT TREE:
{project_tree}

LOGIC SPECIFICATIONS:
{LOGIC_SPECS}

UI SPECIFICATIONS:
{UI_SPECS}

SPRINT GOAL:
{sprint_goal}

Important orchestration requirement:
- The JSON task list must use file paths relative to the target app root.
- Example file_name values: Pages/Index.cshtml, Pages/Index.cshtml.cs, wwwroot/css/zen-green.css, Models/product.cs.
- Do not write code. Produce a human-readable manifest and a JSON task list only.
"""
    manifest = call_agent(ARCHITECT_PROMPT, user_prompt, temperature=0.15)
    tasks = extract_tasks_from_manifest(manifest)

    MANIFEST_PATH.write_text(manifest, encoding="utf-8")
    TASKS_PATH.write_text(json.dumps({"tasks": tasks}, indent=2), encoding="utf-8")

    return manifest, tasks


In [ ]:
# Step 7: Coder and auditor loop

def build_coder_prompt(relative_path: Path, task_description: str, previous_feedback: str = "") -> str:
    existing_content = read_existing_target_file(relative_path)
    project_tree = get_project_tree(TARGET_APP_PATH)

    return f"""
Generate the complete content for this target file:

FILE PATH:
{relative_path.as_posix()}

TASK DESCRIPTION:
{task_description}

CURRENT TARGET PROJECT TREE:
{project_tree}

EXISTING FILE CONTENT, IF ANY:
{existing_content if existing_content else "[File does not currently exist.]"}

LOGIC SPECIFICATIONS:
{LOGIC_SPECS}

UI SPECIFICATIONS:
{UI_SPECS}

PREVIOUS AUDIT FEEDBACK TO FIX:
{previous_feedback if previous_feedback else "[None]"}

Return only the raw file content. Do not include markdown fences or explanations.
"""


def audit_generated_file(relative_path: Path, code_text: str, task_description: str) -> tuple[str, str]:
    logic_prompt = f"""
Review this generated file for logic, data, security, compilation, and architecture risks.

FILE PATH:
{relative_path.as_posix()}

TASK DESCRIPTION:
{task_description}

LOGIC SPECIFICATIONS:
{LOGIC_SPECS}

CODE TO REVIEW:
{code_text}
"""

    ui_prompt = f"""
Review this generated file for UI/UX, Zen Green theme, English-only UI, responsive behavior, tooltips, alignment, and validation display rules.

FILE PATH:
{relative_path.as_posix()}

TASK DESCRIPTION:
{task_description}

UI SPECIFICATIONS:
{UI_SPECS}

CODE TO REVIEW:
{code_text}
"""

    logic_feedback = call_agent(LOGIC_TESTER_PROMPT, logic_prompt, temperature=0.0)
    ui_feedback = call_agent(UI_AUDITOR_PROMPT, ui_prompt, temperature=0.0)
    return logic_feedback, ui_feedback


def orchestrate_file_build(task: dict, max_attempts: int = 2) -> dict:
    """Generate and audit one file. Save to outputs/generated_files only if both audits pass."""
    relative_path = safe_relative_path(task["file_name"])
    task_description = task["task_description"]

    print(f"
Building: {relative_path.as_posix()}")
    previous_feedback = ""

    for attempt in range(1, max_attempts + 1):
        print(f"Attempt {attempt}/{max_attempts}")

        coder_prompt = build_coder_prompt(relative_path, task_description, previous_feedback)
        generated_code = call_agent(CODER_PROMPT, coder_prompt, temperature=0.2)
        generated_code = strip_markdown_code_fence(generated_code)

        logic_feedback, ui_feedback = audit_generated_file(relative_path, generated_code, task_description)
        logic_passed = starts_with_pass(logic_feedback)
        ui_passed = starts_with_pass(ui_feedback)

        record = {
            "task_id": task.get("task_id"),
            "file_name": relative_path.as_posix(),
            "attempt": attempt,
            "logic_passed": logic_passed,
            "ui_passed": ui_passed,
            "logic_feedback": logic_feedback,
            "ui_feedback": ui_feedback,
        }
        append_audit_log(record)

        if logic_passed and ui_passed:
            saved_path = save_generated_file(relative_path, generated_code)
            print(f"PASS: saved generated file to {saved_path}")
            return {**record, "status": "PASS", "generated_path": str(saved_path)}

        print("FAILED AUDIT")
        if not logic_passed:
            print("Logic feedback:")
            print(logic_feedback)
        if not ui_passed:
            print("UI feedback:")
            print(ui_feedback)

        previous_feedback = f"""
Logic feedback:
{logic_feedback}

UI feedback:
{ui_feedback}
"""

    print(f"FAILED after {max_attempts} attempts: {relative_path.as_posix()}")
    return {**record, "status": "FAIL"}


In [ ]:
# Step 8: Apply generated files to the target app after human approval

def list_generated_files() -> list[Path]:
    return sorted([p for p in GENERATED_DIR.rglob("*") if p.is_file()])


def apply_generated_files_to_target():
    generated_files = list_generated_files()
    if not generated_files:
        print("No generated files to apply.")
        return

    print("Generated files ready to apply:")
    for p in generated_files:
        rel = p.relative_to(GENERATED_DIR)
        print(" -", rel.as_posix())

    approval = input("Apply these files to the target app repo? (y/n): ").strip().lower()
    if approval != "y":
        print("Apply cancelled.")
        return

    for src in generated_files:
        rel = src.relative_to(GENERATED_DIR)
        dest = TARGET_APP_PATH / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(src, dest)
        print(f"Applied: {rel.as_posix()}")

    print("Done. Review git diff in the target app repo.")


In [ ]:
# Step 9: Build and Git helpers

def run_command(command: list[str], cwd: Path) -> subprocess.CompletedProcess:
    print("Running:", " ".join(command))
    result = subprocess.run(command, cwd=str(cwd), capture_output=True, text=True, shell=False)
    print("STDOUT:")
    print(result.stdout)
    if result.stderr:
        print("STDERR:")
        print(result.stderr)
    print("Return code:", result.returncode)
    return result


def dotnet_build():
    return run_command(["dotnet", "build"], TARGET_APP_PATH)


def git_diff():
    return run_command(["git", "diff", "--stat"], TARGET_APP_PATH)


def git_status():
    return run_command(["git", "status", "--short"], TARGET_APP_PATH)


In [ ]:
# Step 10: Full sprint workflow

def start_agentic_workflow(sprint_goal: str, max_attempts_per_file: int = 2):
    print("Architect is preparing the sprint plan...")
    manifest, tasks = call_architect(sprint_goal)

    print("" + "=" * 80)
    print("ARCHITECT MANIFEST")
    print("=" * 80)
    print(manifest)
    print("=" * 80)

    if not tasks:
        print("No valid tasks were found. Check the Architect prompt or JSON output.")
        return []

    print(f"Architect proposed {len(tasks)} tasks:")
    for task in tasks:
        print(f"{task['task_id']}. {task['file_name']}")

    approval = input("Proceed with code generation? (y/n): ").strip().lower()
    if approval != "y":
        print("Sprint cancelled before code generation.")
        return []

    results = []
    for task in tasks:
        result = orchestrate_file_build(task, max_attempts=max_attempts_per_file)
        results.append(result)

    passed = sum(1 for r in results if r.get("status") == "PASS")
    failed = sum(1 for r in results if r.get("status") != "PASS")
    print(f"Sprint generation complete. PASS: {passed}, FAIL: {failed}")
    print(f"Generated files are in: {GENERATED_DIR}")
    return results


## Step 11: Sprint 1 goal

Run the next cell to start Sprint 1. The workflow will:

1. Ask the Architect for a manifest and JSON task list.
2. Ask for your approval.
3. Generate each file.
4. Audit each file with the Logic Tester and UI Auditor.
5. Save passing files to `outputs/generated_files/`.

After that, run `apply_generated_files_to_target()` to copy approved files into the ASP.NET Core app repo.


In [ ]:
SPRINT_1_GOAL = """
Sprint 1: Infrastructure and app shell.

Build the English-only ASP.NET Core Razor Pages app shell for the ERP Invoice application.
Create or update only the files needed for:

- global Zen Green CSS tokens and common UI classes
- main layout with English-only menu placeholders
- landing page with welcome message
- login placeholder with database selector, username, password, remember-me, and sign-in button
- menu placeholders for Master Data, Transactions, Reports, and Control

Do not implement full authentication yet. Do not implement product, customer, or invoice CRUD yet.
All UI text must be English only.
"""

# Uncomment to run Sprint 1.
# results = start_agentic_workflow(SPRINT_1_GOAL, max_attempts_per_file=2)


## Step 12: Apply, build, and inspect

After the generated files pass audit, run these cells manually.

This is intentionally human-in-the-loop. We are building software, not summoning a chaos demon with write access.


In [ ]:
# Apply approved generated files to the target app repo.
# apply_generated_files_to_target()


In [ ]:
# Build the target ASP.NET Core application.
# dotnet_build()


In [ ]:
# Inspect Git status and diff summary in the target app repo.
# git_status()
# git_diff()
